# RAG Query Pipeline

Run queries through the full **Hybrid RAG** system:

1. **Query Router** — classifies intent → SQL / Vector / Web / Hybrid
2. **Retrieval** — Dense (BGE-M3) + BM25 → RRF Fusion → Jina Reranker
3. **Agent** — LangGraph StateGraph with CRAG (Corrective RAG) guardrails
4. **Generation** — Groq LLaMA 3.3-70B (with fallback chain)

---

In [1]:
import sys
import os
from pathlib import Path
from _setup import PROJECT_ROOT

print(f'Project root: {PROJECT_ROOT}')
print(f"GROQ_API_KEY set: {'[OK]' if os.getenv('GROQ_API_KEY') else '[MISSING]'}")
print(f"OPENROUTER_API_KEY set: {'[OK]' if os.getenv('OPENROUTER_API_KEY') else '[MISSING]'}")

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor
GROQ_API_KEY set: [OK]
OPENROUTER_API_KEY set: [OK]


## 1. Query Router

Test the intelligent query router that classifies questions.

In [2]:
from retrieval.query_router import route_query, classify_query

# Test routing with different query types
test_queries = [
    "What is the price of AAPL?",           # → SQL
    "Quel est le cours d'Attijariwafa?",    # → SQL (BVC)
    "What are the latest Fed interest rate ?",  # → WEB
    "Explain the monetary policy of Bank Al-Maghrib",    # → VECTOR
    "Compare Apple's revenue with Microsoft's",          # → HYBRID
]

print(f"{'Query':<55} {'Route':<8} {'Reason'}")
print("-" * 100)
for q in test_queries:
    result = classify_query(q)
    print(f"{q[:55]:<55} {result['route']:<8} {result.get('reason', '')[:40]}")

Query                                                   Route    Reason
----------------------------------------------------------------------------------------------------
What is the price of AAPL?                              sql      The question is asking for a specific nu
Quel est le cours d'Attijariwafa?                       sql      La question demande un cours spécifique 
[Router] Off-domain entity / live-data marker detected — forcing WEB route
What are the latest Fed interest rate ?                 web      Off-domain finance entity or live-data m
Explain the monetary policy of Bank Al-Maghrib          vector   The question pertains to a macroeconomic
[Router] Off-domain entity / live-data marker detected — forcing WEB route
Compare Apple's revenue with Microsoft's                web      Off-domain finance entity or live-data m


## 2. Vector Search (Semantic Retrieval)

IMPORTANT: This cell loads the BAAI/bge-m3 embedding model (2.2 GB multilingual).
Requires ~4 GB free RAM. If the kernel crashes, skip to section 3 (SQL Search)
which works without the embedding model.

In [3]:
# Uncomment the next line ONLY if you have a lighter multilingual model you want to test.
# The default bge-m3 requires ~4 GB free RAM. Do NOT use bge-small-en (English only).
# os.environ['EMBEDDING_MODEL'] = 'intfloat/multilingual-e5-small'  # 471 MB, multilingual

from retrieval.vector_search import VectorSearch
vs = VectorSearch()

query = "politique monétaire du Maroc"
results = vs.search(query, top_k=3)

print(f"Vector search for: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"--- Result {i} (score: {r['score']:.4f}) ---")
    print(f"Source: {r.get('source_file', r.get('source', 'unknown'))}")
    print(f"Text: {r['text'][:200]}...\n")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector search for: 'politique monétaire du Maroc'

--- Result 1 (score: 0.5929) ---
Source: bank_al_maghrib_PDF_49a32c70.pdf
Text: MAGHRIB - RAPPORT SUR LA POLITIQUE MONÉTAIRE I 9...

--- Result 2 (score: 0.5891) ---
Source: bank_al_maghrib_PDF_262f2ad3.pdf
Text: AL-MAGHRIB - RAPPORT SUR LA POLITIQUE MONÉTAIRE I 27...

--- Result 3 (score: 0.5873) ---
Source: bank_al_maghrib_PDF_56b0315d.pdf
Text: remières et de l’inflation..........................................................................33
3.3 Indice des valeurs unitaires à l’importation du Maroc...........................................



## 3. SQL Search (Structured Data)

In [4]:
from retrieval.sql_search import SQLSearch

sql = SQLSearch()

# Get latest price for a ticker
ticker = "AAPL"
latest = sql.get_latest_price(ticker)
history = sql.get_price_history(ticker, days=5)

print(f"{ticker} — Latest Price:")
if latest:
    print(f"  {sql.format_results(latest)}")

print(f"\n{ticker} — Last 5 Days:")
if history:
    print(f"  {sql.format_results(history)}")

AAPL — Latest Price:
  ticker | date | open | high | low | close | volume
--------------------------------------------------
AAPL | 2026-07-01 | 293.44000244140625 | 296.5899963378906 | 289.20001220703125 | 294.3800048828125 | 50109500

AAPL — Last 5 Days:
  ticker | date | open | high | low | close | volume
--------------------------------------------------
AAPL | 2026-07-01 | 293.44000244140625 | 296.5899963378906 | 289.20001220703125 | 294.3800048828125 | 50109500
AAPL | 2026-06-30 | 281.1700134277344 | 289.94000244140625 | 280.70001220703125 | 289.3599853515625 | 65100200
AAPL | 2026-06-29 | 286.7300109863281 | 288.3699951171875 | 279.8500061035156 | 281.739990234375 | 66427000
AAPL | 2026-06-26 | 275.0 | 285.95001220703125 | 274.2099914550781 | 283.7799987792969 | 261775500
AAPL | 2026-06-25 | 287.3999938964844 | 288.79998779296875 | 273.75 | 275.1499938964844 | 107013700


## 4. Full Agent Query (LangGraph)

IMPORTANT: Loads the embedding model + LLM. Requires ~4 GB free RAM and a valid API key.
If section 2 crashed the kernel, this section will also crash.

In [5]:
from agents.financial_agent import ask

# Simple question
question = "What is the current price of Apple stock?"
print(f"Question: {question}\n")

response = ask(question, history=[], top_n=3)

print(f"Route: {response.get('route', 'unknown')}")
print(f"\nAnswer:\n{response['answer']}")

c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Question: What is the current price of Apple stock?

[Guardrail] Fast-pass: question contains safe financial keywords.
[Router] Off-domain entity / live-data marker detected — forcing WEB route
[WebSearch] Tavily returned 3 results.
[Grader] Context looks sufficient (pre-graded by Jina reranker).


c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\_internal\_runnable.py:400: UserWarning: Parameters {'presence_penalty', 'frequency_penalty'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  ret = self.func(*args, **kwargs)


Route: Web Search

Answer:
The current price of Apple stock is **$295.50**. This value is based on the latest available data, which indicates that the company's market cap stands at **$4.32T**, with a P/E ratio of **35.00** and a dividend yield of **0.36%**. Additionally, it's worth noting that the stock has recently seen an increase of **$5.02** since the market last closed, representing a **1.73%** rise, and has further risen by **$1.72** in pre-market trading. 

Key statistics for Apple stock include:
* Market capitalization: **$4.32T**
* P/E ratio: **35.00**
* Dividend yield: **0.36%**
* Recent price increase: **$5.02** (**1.73%** rise)
* Pre-market trading increase: **$1.72** 

These figures are based on the latest data available and reflect the current market situation for Apple stock.


In [6]:
# Question requiring vector search (Moroccan finance)
question = "Quelles sont les principales mesures de l'AMMC pour protéger les investisseurs?"
print(f"Question: {question}\n")

response = ask(question, history=[], top_n=5)

print(f"Route: {response.get('route', 'unknown')}")
print(f"\nAnswer:\n{response['answer'][:1000]}")
print(f"\nContext used:\n{response.get('context', '')[:500]}")

Question: Quelles sont les principales mesures de l'AMMC pour protéger les investisseurs?

[Guardrail] Fast-pass: question contains safe financial keywords.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[Router] Hybrid retrieval: 35 total docs from 2 streams -> RRF merge
[Router] Reranking 15 hybrid results via Jina API (top_n=5)...
[Router] Best reranker score: 0.6575 (5 docs)
[Grader] Context looks sufficient (pre-graded by Jina reranker).
Route: Vector

Answer:
Les principales mesures de l'AMMC pour protéger les investisseurs incluent :
* Le visa des documents d'information des opérations financières, après examen de la pertinence et de la cohérence de l'information destinée aux investisseurs.
* L'agrément et le visa des notes d'information destinées aux investisseurs pour les produits d'épargne collective tels que les OPCVM, les OPCC, les OPCI et les FPCT.
* Un contrôle permanent sur pièce et sur place des OPCVM, des sociétés de gestion et des dépositaires pour s'assurer qu'ils respectent les dispositions législatives et réglementaires.
* La mise à disposition du public, gratuitement, des documents d'information visés par l'AMMC, préalablement au début de l'opération.
* L'adoption

## 5. Multi-Turn Conversation

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate a conversation with history
history = []

# Turn 1
q1 = "What is NVIDIA's market cap?"
r1 = ask(q1, history=history, top_n=3)
print(f"User: {q1}")
print(f"AI: {r1['answer'][:200]}\n")

# Build history
history.append(HumanMessage(content=q1))
history.append(AIMessage(content=r1['answer']))

# Turn 2 (follow-up)
q2 = "How does it compare to Microsoft?"
r2 = ask(q2, history=history, top_n=3)
print(f"User: {q2}")
print(f"AI: {r2['answer'][:200]}")

[Guardrail] Fast-pass: question contains safe financial keywords.
[Router] Off-domain entity / live-data marker detected — forcing WEB route
[WebSearch] Tavily returned 3 results.
[Grader] Context looks sufficient (pre-graded by Jina reranker).
User: What is NVIDIA's market cap?
AI: **NVIDIA's Market Cap: $4.663 Trillion USD**
NVIDIA's market capitalization is reported to be approximately $4.663 Trillion USD as of June 2026. However, it's worth noting that different sources provi

[Guardrail] Running LLM safety check...
[Guardrail] Is safe? yes
[Contextualizer] Contextualizing query with history...
[Contextualizer] Contextualized to: Comment compare-t-il à Microsoft ?
[Router] Off-domain entity / live-data marker detected — forcing WEB route
[WebSearch] Tavily returned 3 results.
[Grader] Context looks sufficient (pre-graded by Jina reranker).
User: How does it compare to Microsoft?
AI: **NVIDIA's Market Cap Comparison to Microsoft: No Direct Comparison Available**
There is no direct i

## 6. Web Search Route

In [8]:
from retrieval.web_search import WebSearch

web = WebSearch()

query = "Federal Reserve interest rate decision 2024"
results = web.search_financial_news(query, max_results=3)

print(f"Web search: '{query}'\n")
if results:
    print(web.format_results(results))
else:
    print("No results (check SERP API key in .env)")

[WebSearch] Tavily returned 3 results.
Web search: 'Federal Reserve interest rate decision 2024'

## Web Search Results

**1. Stocks fall sharply after Federal Reserve signals fewer rate cuts than expected for 2025 | PBS News**
   The bigger question centers on how much more the Fed will cut next year. A lot is riding on it, particularly after expectations for a series of cuts in 2025 helped the U.S. stock market set an all-time high at least 57 times in 2024.

Fed officials released projections on Wednesday showing the median expectation among them is for two more cuts to the federal funds rate in 2025, or half a percentage point's worth. That's down from the four cuts expected just three months ago.

"We are in a new phase of the process," Fed Chair Jerome Powell said after the central bank quickly eased its main inte
   Source: https://www.pbs.org/newshour/economy/stocks-fall-sharply-after-federal-reserve-signals-fewer-rate-cuts-than-expected-for-2025

**2. How Do Changing Interest 